日化.xlsx 这个数据集是美妆类商品的订单数据，从数量来看，应该是批发类的订单。包含两个 sheet 页（订单表和商品表），可以挖掘的纬度有日期、地区、商品，指标则有销售量、销售额、增长率等。

# 1、数据理解与处理

In [1]:
import pandas as pd 
fact_order = pd.read_excel('../data/日化.xlsx', sheet_name='销售订单表')
dim_product = pd.read_excel('../data/日化.xlsx', sheet_name='商品信息表')

1.1 商品表数据清洗

In [2]:
dim_product.head()

,商品编号,商品名称,商品小类,商品大类,销售单价
0,X001,商品1,面膜,护肤品,121
1,X002,商品2,面膜,护肤品,141
2,X003,商品3,面膜,护肤品,168
3,X004,商品4,面膜,护肤品,211
4,X005,商品5,面膜,护肤品,185


In [3]:
dim_product.describe()

,销售单价
count,122.000000
mean,156.155738
std,58.454619
min,56.000000
25%,102.250000
50%,158.000000
75%,210.750000
max,253.000000


In [4]:
dim_product[dim_product.duplicated()].count()  # 没有完全重复的数据

商品编号    0
商品名称    0
商品小类    0
商品大类    0
销售单价    0
dtype: int64

In [5]:
dim_product[dim_product['商品编号'].duplicated()].count()  # ID 唯一没有重复

商品编号    0
商品名称    0
商品小类    0
商品大类    0
销售单价    0
dtype: int64

In [6]:
dim_product.isnull().sum()   # 没有空值 

商品编号    0
商品名称    0
商品小类    0
商品大类    0
销售单价    0
dtype: int64

1.2 订单表数据清洗

In [7]:
fact_order.head()

,订单编码,订单日期,客户编码,所在区域,所在省份,所在地市,商品编号,订购数量,订购单价,金额
0,D31313,2019-05-16 00:00:00,S22796,东区,浙江省,台州市,X091,892,214,190888.0
1,D21329,2019-05-14 00:00:00,S11460,东区,安徽省,宿州市,X005,276,185,51060.0
2,D22372,2019-08-26 00:00:00,S11101,北区,山西省,忻州市,X078,1450,116,168200.0
3,D31078,2019-04-08 00:00:00,S10902,北区,吉林省,延边朝鲜族自治州,X025,1834,102,187068.0
4,D32470,2019-04-11 00:00:00,S18696,北区,北京市,北京市,X010,887,58,51446.0


In [8]:
fact_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 31452 entries, 0 to 31451
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   订单编码    31452 non-null  str    
 1   订单日期    31452 non-null  object 
 2   客户编码    31452 non-null  str    
 3   所在区域    31450 non-null  str    
 4   所在省份    31450 non-null  str    
 5   所在地市    31452 non-null  str    
 6   商品编号    31451 non-null  str    
 7   订购数量    31450 non-null  object 
 8   订购单价    31448 non-null  object 
 9   金额      31448 non-null  float64
dtypes: float64(1), object(3), str(6)
memory usage: 2.4+ MB


In [9]:
fact_order[fact_order.duplicated()].count()  # 6条完全重复的数据

订单编码    6
订单日期    6
客户编码    6
所在区域    6
所在省份    6
所在地市    6
商品编号    6
订购数量    6
订购单价    6
金额      6
dtype: int64

In [10]:
fact_order.drop_duplicates(inplace=True)   # 删除重复数据
fact_order.reset_index(drop=True, inplace=True)  # 重建索引
fact_order.isnull().sum()  # 查看空值，有几条数据缺失

订单编码    0
订单日期    0
客户编码    0
所在区域    2
所在省份    2
所在地市    0
商品编号    1
订购数量    2
订购单价    4
金额      4
dtype: int64

In [11]:
fact_order = fact_order.ffill().bfill() #空值填充，不创建新对象
print(fact_order.isnull().sum())

订单编码    0
订单日期    0
客户编码    0
所在区域    0
所在省份    0
所在地市    0
商品编号    0
订购数量    0
订购单价    0
金额      0
dtype: int64


In [12]:
fact_order['订单日期'] = fact_order['订单日期'].apply(lambda x: pd.to_datetime(x, format='%Y#%m#%d') if isinstance(x, str) else x)
fact_order[fact_order['订单日期'] > '2026-01-01'] # 有一条脏数据

,订单编码,订单日期,客户编码,所在区域,所在省份,所在地市,商品编号,订购数量,订购单价,金额
20797,D26533,2050-06-09,S21396,北区,河北省,石家庄市,X022,759,158,119922.0


In [13]:
fact_order = fact_order[fact_order['订单日期'] < '2026-01-01'] # 过滤掉脏数据
fact_order['订单日期'].max(), fact_order['订单日期'].min()  # 数据区间在 2019-01-01 到 2019-09-30 之间

(Timestamp('2019-09-30 00:00:00'), Timestamp('2019-01-01 00:00:00'))

In [14]:
fact_order['订购数量'] = fact_order['订购数量'].apply(lambda x: x.strip('个') if isinstance(x, str) else x).astype('int')
fact_order['订购单价'] = fact_order['订购单价'].apply(lambda x: x.strip('元') if isinstance(x, str) else x).astype('float')
fact_order['金额'] = fact_order['金额'].astype('float')

In [15]:
fact_order.info()

<class 'pandas.DataFrame'>
Index: 31445 entries, 0 to 31445
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   订单编码    31445 non-null  str           
 1   订单日期    31445 non-null  datetime64[us]
 2   客户编码    31445 non-null  str           
 3   所在区域    31445 non-null  str           
 4   所在省份    31445 non-null  str           
 5   所在地市    31445 non-null  str           
 6   商品编号    31445 non-null  str           
 7   订购数量    31445 non-null  int64         
 8   订购单价    31445 non-null  float64       
 9   金额      31445 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(6)
memory usage: 2.6 MB


In [16]:
fact_order['所在省份'] = fact_order['所在省份'].str.replace('自治区|维吾尔|回族|壮族|省|市', '',regex=True)  # 对省份做个清洗，便于可视化
fact_order['所在省份'].unique()

<StringArray>
[ '浙江',  '安徽',  '山西',  '吉林',  '北京',  '云南',  '广东',  '广西', '内蒙古',  '新疆',  '湖北',
  '江苏',  '甘肃',  '四川',  '河南',  '福建',  '陕西',  '辽宁',  '山东',  '江西',  '重庆',  '河北',
  '湖南',  '上海',  '贵州',  '天津',  '海南',  '宁夏', '黑龙江']
Length: 29, dtype: str

In [17]:
fact_order['编码长度'] = fact_order['客户编码'].astype(str).str.len()
print(fact_order['编码长度'].value_counts())#字符长度分布

# 找出长度不同的记录
主流长度 = fact_order['编码长度'].mode()[0]  # 出现最多的长度
异常数据 = fact_order[fact_order['编码长度'] != 主流长度]
print(异常数据[['客户编码', '编码长度']].drop_duplicates('客户编码'))

编码长度
6    31444
8        1
Name: count, dtype: int64
          客户编码  编码长度
8663  编号S13676     8


In [18]:
fact_order['客户编码'] = fact_order['客户编码'].str.replace('编号', '') #有一条不一样结构的数据

# 2、数据分析与可视化


2.1 每月订购情况

In [19]:
from pyecharts import options as opts
from pyecharts.charts import Map, Bar, Line
from pyecharts.components import Table
from pyecharts.options import ComponentTitleOpts
from pyecharts.faker import Faker

fact_order['订单月份'] = fact_order['订单日期'].apply(lambda x: x.month) 
item = fact_order.groupby('订单月份').agg({'订购数量': 'sum', '金额': 'sum'}).to_dict()
x = [f'{key} 月' for key in item['订购数量'].keys()]
y1 = [round(val/10000, 2) for val in item['订购数量'].values()]
y2 = [round(val/10000/10000, 2) for val in item['金额'].values()]
c = (
    Bar()
    .add_xaxis(x)
    .add_yaxis("订购数量（万件）", y1)
    .add_yaxis("金额（亿元）", y2)
    .set_global_opts(title_opts=opts.TitleOpts(title="每月订购情况"))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=True),
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=True),
    )    
)
c.render('07.html')#新版jupyter又渲染不出来了，直接到处查看

'D:\\8102240429\\E-Commerce-Data-Analysis\\part3_daily_chemical\\07.html'

2.2 哪里的人最爱美（订购数量排行）

In [20]:
item = fact_order.groupby('所在地市').agg({'订购数量': 'sum'}).sort_values(by='订购数量', ascending=False)[:20].sort_values(by='订购数量').to_dict()['订购数量']

c = (
    Bar()
    .add_xaxis([*item.keys()])
    .add_yaxis("订购量", [round(v/10000, 2) for v in item.values()], label_opts=opts.LabelOpts(position="right", formatter='{@[1]/} 万'))
    .reversal_axis()
    .set_global_opts(
        title_opts=opts.TitleOpts("订购数量排行 TOP20")
    )
)
c.render('08.html')

'D:\\8102240429\\E-Commerce-Data-Analysis\\part3_daily_chemical\\08.html'

2.3 什么类型的美妆需求量最大

In [21]:
order = pd.merge(fact_order, dim_product, on='商品编号',how='inner')  # 表关联
order

,订单编码,订单日期,客户编码,所在区域,所在省份,所在地市,商品编号,订购数量,订购单价,金额,编码长度,订单月份,商品名称,商品小类,商品大类,销售单价
0,D31313,2019-05-16,S22796,东区,浙江,台州市,X091,892,214.0,190888.0,6,5,商品91,粉底,彩妆,214
1,D21329,2019-05-14,S11460,东区,安徽,宿州市,X005,276,185.0,51060.0,6,5,商品5,面膜,护肤品,185
2,D22372,2019-08-26,S11101,北区,山西,忻州市,X078,1450,116.0,168200.0,6,8,商品78,口红,彩妆,116
3,D31078,2019-04-08,S10902,北区,吉林,延边朝鲜族自治州,X025,1834,102.0,187068.0,6,4,商品25,眼霜,护肤品,102
4,D32470,2019-04-11,S18696,北区,北京,北京市,X010,887,58.0,51446.0,6,4,商品10,面膜,护肤品,58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31439,D26211,2019-03-03,S14926,北区,辽宁,大连市,X014,1166,86.0,100276.0,6,3,商品14,面膜,护肤品,86
31440,D23434,2019-09-27,S18445,北区,河北,沧州市,X011,976,226.0,220576.0,6,9,商品11,面膜,护肤品,226
31441,D27724,2019-08-06,S15803,西区,贵州,黔东南州,X038,665,102.0,67830.0,6,8,商品38,面霜,护肤品,102
31442,D38486,2019-09-15,S12592,东区,浙江,杭州市,X038,665,102.0,67830.0,6,9,商品38,面霜,护肤品,102


In [22]:
order.groupby(['商品大类','商品小类']).agg({'订购数量': 'sum'}).sort_values(by=['商品大类', '订购数量'], ascending=[True, False])
#商品大类升序，小类订购数量降序

订购数量
商品大类 商品小类         
彩妆   口红    2013024
     粉底    1188621
     睫毛膏    587399
     眼影     296599
     蜜粉      45534
护肤品  面膜    5451914
     面霜    4567570
     爽肤水   3523687
     眼霜    3350078
     隔离霜   2488124
     防晒霜   2388610
     洁面乳   1928020

2.4 哪些省份的美妆需求量最大

In [24]:
item = fact_order.groupby('所在省份').agg({'订购数量': 'sum'}).to_dict()['订购数量']
name_map = {
    "上海":"上海市","云南":"云南省","内蒙古":"内蒙古自治区","北京":"北京市",
    "吉林":"吉林省","四川":"四川省","天津":"天津市","宁夏":"宁夏回族自治区",
    "安徽":"安徽省","山东":"山东省","山西":"山西省","广东":"广东省",
    "广西":"广西壮族自治区","新疆":"新疆维吾尔自治区","江苏":"江苏省","江西":"江西省",
    "河北":"河北省","河南":"河南省","浙江":"浙江省","海南":"海南省",
    "湖北":"湖北省","湖南":"湖南省","甘肃":"甘肃省","福建":"福建省",
    "西藏":"西藏自治区","贵州":"贵州省","辽宁":"辽宁省","重庆":"重庆市",
    "陕西":"陕西省","青海":"青海省","黑龙江":"黑龙江省","台湾":"台湾省"
}
temp_data = {name_map[k]: v for k, v in item.items()}#这里还是得做一下映射，不然图片上是灰的
#key是拿的全称，values是拿的汇总后的订单总数
c = (
    Map()
    .add("订购数量", [*temp_data.items()], "china", is_map_symbol_show=False)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True))
    .set_global_opts(
        title_opts=opts.TitleOpts(title='省份分布'),
        visualmap_opts=opts.VisualMapOpts(max_=1000000),            
    )
)
c.render('09.html')

'D:\\8102240429\\E-Commerce-Data-Analysis\\part3_daily_chemical\\09.html'

2.5 通过 RFM 模型挖掘客户价值

RFM 模型是衡量客户价值和客户创利能力的重要工具和手段，其中由3个要素构成了数据分析最好的指标，分别是：

R-Recency（最近一次购买时间）
F-Frequency（消费频率）
M-Money（消费金额）
设定一个计算权重，比如 R-Recency 20% F-Frequency 30% M-Money 50% ，最后通过这个权重进行打分，量化客户价值，后续还可以基于分数进一步打标签，用来指导二次营销的策略。

In [25]:
data_rfm = fact_order.groupby('客户编码').agg({'订单日期': 'max', '订单编码': 'count', '金额': 'sum'})
data_rfm.columns = ['最近一次购买时间', '消费频率', '消费金额']

In [26]:
data_rfm['R'] = data_rfm['最近一次购买时间'].rank(pct=True)   # 转化为排名 百分比，便于后续切片
data_rfm['F'] = data_rfm['消费频率'].rank(pct=True)
data_rfm['M'] = data_rfm['消费金额'].rank(pct=True)
data_rfm.sort_values(by='R', ascending=False)  

,最近一次购买时间,消费频率,消费金额,R,F,M
客户编码,,,,,,
S10451,2019-09-30,34,5482721.0,0.980148,0.654663,0.731302
S22925,2019-09-30,31,3449117.0,0.980148,0.591413,0.457987
S10469,2019-09-30,30,4198071.0,0.980148,0.570175,0.564174
S19512,2019-09-30,18,3021545.0,0.980148,0.282548,0.370268
S19485,2019-09-30,9,1463984.0,0.980148,0.060942,0.108957
...,...,...,...,...,...,...
S16503,2019-04-07,14,1682893.0,0.004617,0.198061,0.146814
S20864,2019-03-14,8,1118752.0,0.003232,0.039243,0.047091
S17547,2019-03-14,10,1784531.0,0.003232,0.087258,0.163435


In [27]:
data_rfm['score'] = data_rfm['R'] * 20 + data_rfm['F'] * 30 + data_rfm['M'] * 50
data_rfm['score'] = data_rfm['score'].round(1)
data_rfm.sort_values(by='score', ascending=False)  

,最近一次购买时间,消费频率,消费金额,R,F,M,score
客户编码,,,,,,,
S17476,2019-09-30,69,10325832.0,0.980148,0.986611,0.987073,98.6
S22326,2019-09-30,62,10074609.0,0.980148,0.973223,0.984303,98.0
S11581,2019-09-28,79,10333668.0,0.918283,0.996768,0.987996,97.7
S12848,2019-09-29,66,9673572.0,0.944598,0.980609,0.980609,97.3
S19095,2019-09-26,81,11031632.0,0.864728,0.999077,0.996307,97.1
...,...,...,...,...,...,...,...
S12690,2019-05-07,7,917233.0,0.012927,0.022622,0.024931,2.2
S11176,2019-06-09,7,614134.0,0.036011,0.022622,0.009234,1.9
S18379,2019-07-05,4,400195.0,0.071099,0.003232,0.004617,1.7


根据这个分数结果，我们可以对客户打上一些标签，比如大于 80 分的，标志为优质客户，在资源有限的情况下，可以优先服务好优质客户。